# Camera-trap live demonstration
Run both cells in a fresh Colab runtime with the team Drive mounted. Choose CCT20 test images or team test images.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CameraTrap_Phase2')
PROJECT = Path.cwd()
if not (PROJECT/'runtime/live.py').is_file():
    PROJECT = Path(tempfile.mkdtemp())/'project'
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','project','https://github.com/salehdev3/Capstone_5.git',str(PROJECT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)
sys.path.insert(0,str(PROJECT/'runtime'))
from live import cases, execute, initialize, video_cases, team_test_cases, cct_test_cases
pipeline, decision = initialize(ROOT)
print('Loaded integrated release:', pipeline.policy_sha256)


In [ ]:
import pandas as pd
from IPython.display import display, Image as DisplayImage

DEMO_INPUT = 'cct_test' #@param ['cct_test','team_test']
batch = cct_test_cases(ROOT) if DEMO_INPUT == 'cct_test' else team_test_cases(ROOT)
print('Images selected:', len(batch))

dest, result = execute(pipeline, batch, ROOT/'presentation_demos')
display(pd.DataFrame(result['events']))
display(pd.DataFrame([
    {'id': frame['id'], 'animals': frame['count'],
     'inference_seconds': frame['timing']['total']}
    for frame in result['frames']
]))
for index, frame in enumerate(result['frames'], 1):
    display(DisplayImage(filename=str(dest/f'frame_{index:02}.jpg'), width=720))
print('HTTP response and recorded replay:', dest)
